In [ ]:
#create a gaussian with a cutoff at a given radius
import numpy as np
import matplotlib.pyplot as plt
def gaussian_with_cutoff(x, sigma, cutoff):
    """
    Create a Gaussian function with a cutoff at a specified radius.
    
    Parameters:
    x (numpy.ndarray): The input array of distances.
    sigma (float): The standard deviation of the Gaussian.
    cutoff (float): The cutoff radius beyond which the Gaussian is zero.
    
    Returns:
    numpy.ndarray: The Gaussian values with cutoff applied.
    """
    gaussian = np.exp(-0.5 * (x / sigma) ** 2)
    gaussian[x > cutoff] = 0
    return gaussian
def plot_gaussian_with_cutoff(sigma, cutoff):
    """Plot the Gaussian function with a cutoff.    
    Parameters:
    sigma (float): The standard deviation of the Gaussian.
    cutoff (float): The cutoff radius beyond which the Gaussian is zero.
    """ 

    x = np.linspace(-4 * cutoff, 4 * cutoff, 1000)
    y = gaussian_with_cutoff(x, sigma, cutoff)
    
    plt.figure(figsize=(8, 5))
    plt.plot(x, y, label=f'Sigma: {sigma}, Cutoff: {cutoff}')
    # plt.title('Gaussian Function with Cutoff')
    plt.xlabel('Distance')
    plt.ylabel('Gaussian Value')
    # plt.axvline(cutoff, color='red', linestyle='--', label='Cutoff Radius')
    # plt.legend()
    # plt.grid()
    plt.show()

if __name__ == "__main__":
    # Example parameters
    sigma = 0.5
    cutoff = 0.4
    
    # Plot the Gaussian with cutoff
    plot_gaussian_with_cutoff(sigma, cutoff)

In [ ]:
import scipy
from jax import vmap
from scipy.integrate import cumulative_trapezoid

def get_sigma_from_integral(p, Sigma, negloglike_all, nparams, max_fun_params, fop, number_points=10**3):
        #j is the index of the parameter we are looking at
        #i is the index of the data point we are looking at

        def fraction_likelihood(x, p, j, factor, log=True):

            def compute_negloglike(params):

                return jnp.abs(fop(params) - negloglike_all - jnp.log(1e3))
            def inf_value(_):
                return 0.0999
            
            # Modify parameter
            params = jax.lax.cond(log,
                  lambda _: p.at[j].set(p[j] + factor * 10**x[0]),
                  lambda _: p.at[j].set(p[j] + factor * x[0]),            
                  operand=None)
            
            negloglike = jax.lax.cond(jnp.isinf(fop(params)), inf_value, compute_negloglike, params)
            return negloglike

        def get_loss():
            return jax.jit(fraction_likelihood)

        def wrap_loss(fraction_likelihood):
            def loss(x, p, j, factor, log=True):
                # print(fraction_likelihood(x, p, j, factor))
                return fraction_likelihood(x, p, j, factor, log)
            

            return loss
        
        def get_boundary(res, log=True):
            if log:
                return 10**res.x[0]
            else:
                return res.x[0]
        
        def get_integral(theta, boundary_left, boundary_right, param, fop, negloglike, number_points=5*10**2):
            def compute_like(theta):
                return jnp.exp(-fop(theta) + negloglike)

            max_boundary = np.max([boundary_left, boundary_right])
            boundary_range = np.array([-boundary, boundary])

            boundaries = [boundary_left, boundary_right]
            # Loop through left and right boundaries
            integrals = []
            for index_boundary, boundary_shift in enumerate(boundary_range):
            # for boundary_shift in boundary_range:
                a_range = np.linspace(theta, theta + boundary_shift, number_points)
                thetas = np.tile(param, (len(a_range), 1))
                thetas[:, j] = a_range
                like = vmap(compute_like)(thetas)
                # print(boundaries[index_boundary])
                # print(a_range)
                # print('theta', theta)
                like = jnp.where(np.abs(theta - a_range) > boundaries[index_boundary], 0, like)
                # Integrate
                # integral = scipy.integrate.cumtrapz(like, a_range)
                integral = cumulative_trapezoid(like, a_range, initial=0)
                integrals.append(integral)

            # Combine left and right integrals
            integral_left, integral_right = [np.array(integral) for integral in integrals]
            # print('integral_left', integral_left, flush=True)
            # print('integral_right', integral_right, flush=True)
            integral = - integral_left + integral_right
            # print('integral', integral, flush=True)

            return integral, a_range
        
        def test_success(res):
            if res.success == False or res.fun > 0.1:
            # if res.success == False or likelihood_diff <= np.log(5*1e2):
                return False
            else:
                return True

        # param = np.append(p[:nparams], p[-2:])
        param = np.append(p[:nparams], p[max_fun_params:])

        for j in range(len(param)):
        # for j in [2]:
                theta = param[j]

                initial_guesses = [-20, -10, -5,-1, 0, 0.5, 1, np.log10(np.abs(theta)), 2, 5]
                initial_guesses = [(val, True) for val in initial_guesses]
                #Plot likelihood
                Delta_plot = 20
                x_range = np.linspace(theta - Delta_plot, theta + Delta_plot, 10**4)
                # x_range = np.append(x_range, theta)
                params_range = np.tile(param, (len(x_range), 1))
                params_range[:, j] = x_range

                nll = []
                # for params in params_range:
                #     # print('params', params, flush=True)
                #     negloglike = fop(params)
                #     # print('negloglike', negloglike, flush=True)
                #     nll = np.append(nll, negloglike)  
                # plt.figure()
                # plt.plot(x_range, np.exp(-nll + jnp.min(nll[np.isnan(nll) == False])))
                # # plt.plot(x_range, np.exp(-nll))
                # plt.yscale('log')
                # # plt.plot(theta_ML, np.exp(-chi2_fcn(theta_ML, xvar, yvar, yerr) + jnp.min(nll)), 'ro')
                # plt.show()
                # sys.exit()

                loss_template = get_loss()
                chi2_fcn = wrap_loss(loss_template)

                for factor in [-1, 1]:
                    print('factor', factor, flush=True)
                    for initial_guess, is_log in initial_guesses:  
                        success = False
                        # print('initial_guess', initial_guess, flush=True)      

                        log = is_log                    
                        res = scipy.optimize.minimize(chi2_fcn, initial_guess, args=(param, j, factor, log), method='Nelder-Mead', tol=1e-8)
                        print('res', res, flush=True)
                        if test_success(res)== True:
                            if factor == -1:
                                boundary_left = get_boundary(res, log)
                                # print('boundary_left', boundary_left, flush=True)
                            else:
                                boundary_right = get_boundary(res, log)
                                # print('boundary_right', boundary_right, flush=True)
                            success = True
                            break
                        # else:
                        #     continue
                    if not success:
                        break


                if not success:
                    # print("Couldn't find integral limits", fcn_i, theta, flush=True)
                    Sigma[j] = np.inf
                    continue

                #do integral
                boundary = np.max([boundary_left, boundary_right])
                integral, a_range = get_integral(theta, boundary_left, boundary_right, param, fop, negloglike_all, number_points=number_points)
                                        
                #get the 68% confidence interval from the integral

                arg_min = np.argmin(abs(0.68 - integral/integral[-1]))
                param68 = a_range[arg_min + 1]
                # print(param68)
                sigma = np.abs(theta - param68)
                Sigma[j] = sigma

        return Sigma, boundary_left, boundary_right, a_range, integral

In [ ]:
import jax.numpy as jnp
import numpy as np

# This is your function fop(params): returns -log L
def gaussian_negloglike(params):
    x = params[0]
    mu = 0.0
    sigma = 1.0
    in_bounds = x <= 0.6
    value = 0.5 * ((x - mu) / sigma) ** 2
    return jnp.where(in_bounds, value, jnp.inf)

In [ ]:
from jax import config
config.update("jax_enable_x64", True)  # Optional for better numerical precision
import jax
import matplotlib.pyplot as plt


#get sigma for the gaussian I have
p = jnp.array([0.0])  # ML estimate at x = 0
Sigma = np.zeros_like(p)
negloglike_all = gaussian_negloglike(p)
nparams = 1
max_fun_params = 1  # No extra params
fop = gaussian_negloglike

sigma_estimate, boundary_left, boundary_right, a_range, integral = get_sigma_from_integral(p, Sigma, negloglike_all, nparams, max_fun_params, fop)
print("Estimated sigma:", sigma_estimate)

print("Boundary left:", boundary_left)
print("Boundary right:", boundary_right)

In [ ]:
#plot gaussian
x = np.linspace(-4, 2, 1000)
y = gaussian_negloglike(jnp.array([x]))
plt.figure(figsize=(8, 5))
plt.plot(x, np.exp(-y + jnp.min(y)), label='Conditional Likelihood', color='black')
plt.axvline(p[0], color='red', linestyle='--', label='ML')
plt.axvline(p[0] + sigma_estimate[0], color='blue', linestyle='--', label=r"$\sigma$")
plt.axvline(p[0] - sigma_estimate[0], color='blue', linestyle='--')
plt.xlabel('x')
plt.ylabel('Likelihood')
plt.legend()
plt.show()  


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(-3, 2, 1000)
y = gaussian_negloglike(jnp.array([x]))
likelihood = np.exp(-y + jnp.min(y))
cm = plt.get_cmap('Set1')

plt.figure(figsize=(8, 5))
plt.plot(x, likelihood, color='black')
plt.axvline(p[0], color=cm(0), linestyle='--', label='ML', lw=2.5)
plt.axvline(p[0] + sigma_estimate[0], color=cm(1), linestyle='--', label=r"$\mu \pm \hat{\sigma}$", lw=2.5)
plt.axvline(p[0] - sigma_estimate[0], color=cm(1), linestyle='--', lw=2.5)
plt.fill_between(x, 0, likelihood, where=((x > p[0] - sigma_estimate[0]) & (x < p[0] + sigma_estimate[0])), color=cm(1), alpha=0.1, label='68% CI')

plt.xlabel(r"Parameter $\theta_i$", fontsize=18)
plt.ylabel(r"Likelihood $\mathcal{L}(\theta_i|\theta_{j\ne i})$", fontsize=18)
# plt.title("Uncertainty from Conditional Likelihood")
plt.ylim(bottom=0)
plt.legend(fontsize=16)
#make ticks size 14
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
# plt.grid(True)
plt.tight_layout()
plt.savefig("conditional_likelihood_uncertainty.png", dpi=300)
plt.show()